In [1]:
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.3/251.3 kB 2.8 MB/s eta 0:00:0000:01


In [2]:
!pip install anndata==0.8.0

  Using cached anndata-0.8.0-py3-none-any.whl (96 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.8 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: h5py
    Found existing installation: h5py 2.10.0
    Uninstalling h5py-2.10.0:
      Successfully uninstalled h5py-2.10.0
  Attempting uninstall: anndata
    Found existing installation: anndata 0.7.8
    Uninstalling anndata-0.7.8:
      Successfully uninstalled anndata-0.7.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
samap 1.0.2 requires h5py<=2.10, but you have h5py 3.8.0 which is incompatible.
sam-algorithm 1.0.0 requires h5py<=2.10.0, but you have h5py 3.8.0 which is incompatible.


In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
from multiprocessing import Pool
from joblib import Parallel, delayed
import itertools

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
with open('parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [20]:
fn = 'Active_SAM_joined/SAM_Allen_Institute_allhypo_01272026.h5ad'

In [21]:
sam=SAM()
sam.load_data(fn)
gene_dict = {}
for i in range(len(sam.adata.var_names)):
    gene_dict[sam.adata.var_names[i]] = i

In [22]:
sam.adata.obs.columns

Index(['Unnamed: 0', 'cell_label', 'cell_barcode',
       'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label',
       'entity', 'brain_section_label', 'library_method',
       'region_of_interest_acronym', 'donor_label', 'donor_genotype',
       'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias',
       'subclass_id_label', 'supertype_id_label', 'class_id_label'],
      dtype='object')

In [25]:
level = 'subclass_id_label'

In [26]:
for item in sam.adata.obs[level].unique():
    if item not in parent_dict.keys():
        parent_dict[item] = 'hypo'
    if item[:2] == 'mo':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mg':
        parent_dict[item] = 'hypo'
    if item[:2] == 'xt':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ac':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cj':
        parent_dict[item] = 'hypo'
    if item[:2] == 'dr':
        parent_dict[item] = 'hypo'
    if item[:2] == 'mv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'cc':
        parent_dict[item] = 'hypo'
    if item[:2] == 'rv':
        parent_dict[item] = 'hypo'
    if item[:2] == 'ri':
        parent_dict[item] = 'hypo'
    if item[-2:] == 'NN':
        parent_dict[item] = 'Non neuron'

In [27]:
parent_dict['not hypo'] = 'not hypo'
parent_dict['hypo'] = 'hypo'
parent_dict['Unknown'] = 'hypo'
parent_dict['Unknwon_2'] = 'hypo'
parent_dict['Non neuron'] = 'not hypo'

In [28]:
ordered_genes = sam.identify_marker_genes_ratio(level)

In [29]:
ordered_genes.keys()

dict_keys(['066 NDB-SI-ant Prdm12 Gaba', '073 MEA-BST Sox6 Gaba', '074 MEA-BST Lhx6 Sp9 Gaba', '075 MEA-BST Lhx6 Nr2e1 Gaba', '076 MEA-BST Lhx6 Nfib Gaba', '077 CEA-BST Gal Avp Gaba', '078 SI-MA-ACB Ebf1 Bnc2 Gaba', '079 CEA-BST Six3 Cyp26b1 Gaba', '080 CEA-AAA-BST Six3 Sp9 Gaba', '081 ACB-BST-FS D1 Gaba', '082 CEA-BST Ebf1 Pdyn Gaba', '083 CEA-BST Rai14 Pdyn Crh Gaba', '084 BST-SI-AAA Six3 Slc22a3 Gaba', '085 SI-MPO-LPO Lhx8 Gaba', '086 MPO-ADP Lhx8 Gaba', '087 MPN-MPO-LPO Lhx6 Zfhx3 Gaba', '088 BST Tac2 Gaba', '089 PVR Six3 Sox3 Gaba', '090 BST-MPN Six3 Nrgn Gaba', '091 ARH-PVi Six6 Dopa-Gaba', '092 TMv-PMv Tbx3 Hist-Gaba', '093 RT-ZI Gnb3 Gaba', '094 SCH Six6 Cdc14a Gaba', '095 DMH Prdm13 Gaba', '096 PVHd Gsc Gaba', '097 PVHd-SBPV Six3 Prox1 Gaba', '098 AHN-SBPV-PVHd Pdrm12 Gaba', '099 SBPV-PVa Six6 Satb2 Gaba', '100 AHN Onecut3 Gaba', '101 ZI Pax6 Gaba', '102 DMH-LHA Gsx1 Gaba', '103 PVHd-DMH Lhx6 Gaba', '104 TU-ARH Otp Six6 Gaba', '105 TMd-DMH Foxd2 Gaba', '106 PVpo-VMPO-MPN Hmx2 

In [30]:
hypo_ct = [i for i in sam.adata.obs[level].unique() if parent_dict[parent_dict[i]] == 'hypo']

In [31]:
np.sort(hypo_ct)

array(['066 NDB-SI-ant Prdm12 Gaba', '073 MEA-BST Sox6 Gaba',
       '074 MEA-BST Lhx6 Sp9 Gaba', '075 MEA-BST Lhx6 Nr2e1 Gaba',
       '076 MEA-BST Lhx6 Nfib Gaba', '077 CEA-BST Gal Avp Gaba',
       '078 SI-MA-ACB Ebf1 Bnc2 Gaba', '079 CEA-BST Six3 Cyp26b1 Gaba',
       '080 CEA-AAA-BST Six3 Sp9 Gaba', '081 ACB-BST-FS D1 Gaba',
       '082 CEA-BST Ebf1 Pdyn Gaba', '083 CEA-BST Rai14 Pdyn Crh Gaba',
       '084 BST-SI-AAA Six3 Slc22a3 Gaba', '085 SI-MPO-LPO Lhx8 Gaba',
       '086 MPO-ADP Lhx8 Gaba', '087 MPN-MPO-LPO Lhx6 Zfhx3 Gaba',
       '088 BST Tac2 Gaba', '089 PVR Six3 Sox3 Gaba',
       '090 BST-MPN Six3 Nrgn Gaba', '091 ARH-PVi Six6 Dopa-Gaba',
       '092 TMv-PMv Tbx3 Hist-Gaba', '093 RT-ZI Gnb3 Gaba',
       '094 SCH Six6 Cdc14a Gaba', '095 DMH Prdm13 Gaba',
       '096 PVHd Gsc Gaba', '097 PVHd-SBPV Six3 Prox1 Gaba',
       '098 AHN-SBPV-PVHd Pdrm12 Gaba', '099 SBPV-PVa Six6 Satb2 Gaba',
       '100 AHN Onecut3 Gaba', '101 ZI Pax6 Gaba',
       '102 DMH-LHA Gsx1 Gaba', '10

In [103]:
def markers_celltype(cto, test_top, diff, difftwo, qdiffth, A, obs_level, ordered_genes, gene_dict):
    mask = obs_level == cto
    ct_X = A[mask, :]
    notct_X = A[~mask, :]
    
    markers = []
    for gene in ordered_genes[cto][:test_top]:
        gene_index = gene_dict[gene]
        ct_exp_g = ct_X[:, gene_index]
        bkgd_exp_g = notct_X[:, gene_index]
        
        fct_exp_g = np.sum(ct_exp_g >= 1) / len(ct_exp_g)
        fbkgd_exp_g = np.sum(bkgd_exp_g >= 1) / len(bkgd_exp_g)
        
        if np.average(ct_exp_g) - np.average(bkgd_exp_g) > diff:
            st, pval = stats.mannwhitneyu(x=ct_exp_g, y=bkgd_exp_g, alternative='greater')
            if pval < .05 / test_top:
                markers.append(gene)
        elif (fct_exp_g - fbkgd_exp_g) / fct_exp_g > qdiffth and fct_exp_g > difftwo:
            st, pval = stats.mannwhitneyu(x=ct_exp_g, y=bkgd_exp_g, alternative='greater')
            if pval < .05 / test_top:
                markers.append(gene)
    obs_level = sam.adata.obs[level].values
    return cto, markers

In [134]:
org = 'CJ'
test_top = 7000
t0 = time.time()
difftwo = .01
query = sam.adata.obs[level].unique()

A = sam.adata.X.A
obs_level = sam.adata.obs[level].values

qdiffth_range = [.8]
diff_range = [1]
print(diff_range)

for qdiffth, diff in itertools.product(qdiffth_range, diff_range):
    print(f'Running qdiffth={qdiffth}, diff={diff}')
    t0 = time.time()
    results = Parallel(n_jobs=4, verbose=10)(
        delayed(markers_celltype)(cto,
                                  test_top=test_top,
                                  diff=diff,
                                  difftwo=difftwo,
                                  qdiffth=qdiffth,
                                  A=A,
                                  obs_level=obs_level,
                                  ordered_genes=ordered_genes,
                                  gene_dict=gene_dict)
        for cto in query
    )
    marker_dict = dict(results)
    
    # Save results
    folder_name = org + '_nonneuron_bkgd_allct_05132026_diff_' + str(diff*100) + 'qdiff_' + str(qdiffth*100) 
    if not os.path.isdir('Important_genes/' + folder_name):
        os.mkdir('Important_genes/' + folder_name)
    df = pd.DataFrame(data = [qdiffth, diff,difftwo], index = ['qdiffth','diff','diff2']) 
    df.to_csv('Important_genes/' + folder_name + '/metadata.csv')
    with open('Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
        pickle.dump(marker_dict, f)
    
    t1 = time.time()
    print(f'qdiffth={qdiffth}, diff={diff} completed in {t1-t0:.1f}s')

[0.5]
Running qdiffth=0.8, diff=0.5


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   12.0s
[Parallel(n_jobs=4)]: Done   9 out of  14 | elapsed:   19.3s remaining:   10.7s
[Parallel(n_jobs=4)]: Done  11 out of  14 | elapsed:   24.3s remaining:    6.6s
[Parallel(n_jobs=4)]: Done  14 out of  14 | elapsed:   34.5s finished


qdiffth=0.8, diff=0.5 completed in 35.1s


In [ ]:
#FOR IDENTIFYINImportant_genes/ENES (ZEBRAFISH SPECIFIC)
test_top = 5000
t0 = time.time()
diff = .01
qdiffth = .8
qdiffth2 = .5
query = hypo_ct
ref = hypo_ct
kn = 10
num_ct_better = 1 
a = 0

marker_dict = {}
allctexp = {}
fin_hist = []
for ct in ref:
    allctexp[ct] = sam.adata[sam.adata.obs[level] == ct].X.A
t1 = time.time()
print('created dictionary in: ' + str(t1 - t0) + ' seconds')

neigh = {}
for q in query:
    dl = []
    for ctt in ref:
        dl.append(distance.euclidean(np.average(sam.adata[sam.adata.obs[level] == q].obsm['X_pca'], axis = 0),np.average(sam.adata[sam.adata.obs[level] == ctt].obsm['X_pca'], axis = 0)))
    dl = np.array(dl)
    ds = np.argsort(dl)
    di = ds[:kn + 1]
    di = di[1:]
    ctn = []
    for item in di:
        ctn.append(ref[item])
    neigh[q] = ctn
t2 = time.time()
print('created neighbors in: ' + str(t2 - t1) + ' seconds')
    
for cto in query:
    print(cto)
    marker_dict[cto] = []
    for gene in ordered_genes[cto][:test_top]:
        num = 0
        neigh_ref = neigh[cto]
        for ctt in neigh_ref:
            ct_exp = allctexp[cto]
            bkgd_exp = allctexp[ctt]
            gene_index = gene_dict[gene]
            bkgd_exp_g = bkgd_exp[:,gene_index]
            ct_exp_g = ct_exp[:,gene_index]
            fct_exp_g = np.sum(ct_exp_g >= 0)/len(ct_exp_g)
            fbkgd_exp_g = np.sum(bkgd_exp_g >= 0)/len(bkgd_exp_g)
            if fct_exp_g > diff and (fct_exp_g-fbkgd_exp_g)/fct_exp_g > qdiffth:
                st, pval = stats.mannwhitneyu(x = ct_exp_g,y = bkgd_exp_g, alternative= 'greater')
                if pval < .05/test_top:
                    num += 1
            elif np.average(ct_exp_g) - np.average(bkgd_exp_g) > qdiffth2:
                st, pval = stats.mannwhitneyu(x = ct_exp_g,y = bkgd_exp_g, alternative= 'greater')
                if pval < .05/test_top:
                    num += 1

        if num >= num_ct_better:
            marker_dict[cto].append(gene)

    print(len(marker_dict[cto]))
    a += 1
    t3 = time.time()
    print(str((a/len(query))*100) + ' percent complete in: ' + str(t3-t2) + ' seconds')

In [32]:
#Check claude 
def markers_celltype_neighbor(cto, neigh_cto, test_top, diff, qdiffth, qdiffth2,
                              num_ct_better, A, obs_level, ordered_genes, gene_dict):
    mask = obs_level == cto
    ct_X = A[mask, :]
    bkgd_all_X = A[np.isin(obs_level, ref) & ~mask,:]                           # all cells NOT in cto
    neigh_X = {ctt: A[obs_level == ctt, :] for ctt in neigh_cto}  # precompute once

    markers = []
    for gene in ordered_genes[cto][:test_top]:
        gene_index = gene_dict[gene]
        ct_exp_g = ct_X[:, gene_index]
        fct_exp_g = np.sum(ct_exp_g > 0) / len(ct_exp_g)
        ct_mean = np.average(ct_exp_g)

        num = 0
        sig = None  # significance vs full background, computed at most once per gene
        for ctt in neigh_cto:
            bkgd_exp_g = neigh_X[ctt][:, gene_index]
            fbkgd_exp_g = np.sum(bkgd_exp_g > 0) / len(bkgd_exp_g)

            # effect size is still measured relative to the close neighbor
            crit = ((fct_exp_g > diff and (fct_exp_g - fbkgd_exp_g) / fct_exp_g > qdiffth)
                    or (ct_mean - np.average(bkgd_exp_g) > qdiffth2))
            if crit:
                if sig is None:
                    st, pval = stats.mannwhitneyu(x=ct_exp_g,
                                                  y=bkgd_all_X[:, gene_index],
                                                  alternative='greater')
                    sig = pval < .05 / test_top
                if sig:
                    num += 1

        if num >= num_ct_better:
            markers.append(gene)
    return cto, markers

In [33]:
#Check claude
test_top = 7000
diff = .01
qdiffth = .8
qdiffth2 = 1
kn = 10
num_ct_better = 1
query = hypo_ct
ref = list(hypo_ct)

A = sam.adata.X.A
obs_level = sam.adata.obs[level].values
pca = sam.adata.obsm['X_pca']

# --- precompute nearest-neighbor cell types once (centroids in PCA space) ---
t0 = time.time()
centroids = np.vstack([pca[obs_level == ct].mean(axis=0) for ct in ref])
neigh = {}
for q in query:
    qc = pca[obs_level == q].mean(axis=0)
    d = np.linalg.norm(centroids - qc, axis=1)
    di = np.argsort(d)[:kn + 1][1:]          # drop self (nearest, distance 0)
    neigh[q] = [ref[i] for i in di]
print('created neighbors in: ' + str(time.time() - t0) + ' seconds')

# --- parallel marker calling ---
t1 = time.time()
results = Parallel(n_jobs=2, verbose=10)(
    delayed(markers_celltype_neighbor)(cto,
                                       neigh[cto],
                                       test_top=test_top,
                                       diff=diff,
                                       qdiffth=qdiffth,
                                       qdiffth2=qdiffth2,
                                       num_ct_better=num_ct_better,
                                       A=A,
                                       obs_level=obs_level,
                                       ordered_genes=ordered_genes,
                                       gene_dict=gene_dict)
    for cto in query
)
marker_dict = dict(results)
print('called markers in: ' + str(time.time() - t1) + ' seconds')
for cto in query:
    print(cto, len(marker_dict[cto]))

created neighbors in: 0.23090124130249023 seconds


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:  3.6min
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:  5.3min
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:  8.4min
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed: 10.9min
[Parallel(n_jobs=2)]: Done  21 tasks      | elapsed: 15.6min
[Parallel(n_jobs=2)]: Done  28 tasks      | elapsed: 19.5min
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed: 25.3min
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed: 30.1min
[Parallel(n_jobs=2)]: Done  57 tasks      | elapsed: 37.9min
[Parallel(n_jobs=2)]: Done  68 tasks      | elapsed: 44.8min
[Parallel(n_jobs=2)]: Done  73 out of  73 | elapsed: 49.7min finished


called markers in: 2983.410032272339 seconds
078 SI-MA-ACB Ebf1 Bnc2 Gaba 327
079 CEA-BST Six3 Cyp26b1 Gaba 519
080 CEA-AAA-BST Six3 Sp9 Gaba 343
104 TU-ARH Otp Six6 Gaba 287
101 ZI Pax6 Gaba 217
066 NDB-SI-ant Prdm12 Gaba 369
090 BST-MPN Six3 Nrgn Gaba 228
089 PVR Six3 Sox3 Gaba 205
073 MEA-BST Sox6 Gaba 369
098 AHN-SBPV-PVHd Pdrm12 Gaba 145
084 BST-SI-AAA Six3 Slc22a3 Gaba 329
096 PVHd Gsc Gaba 242
097 PVHd-SBPV Six3 Prox1 Gaba 171
100 AHN Onecut3 Gaba 196
103 PVHd-DMH Lhx6 Gaba 245
081 ACB-BST-FS D1 Gaba 398
113 MEA-COA-BMA Ccdc42 Glut 391
119 SI-MA-LPO-LHA Skor1 Glut 310
114 COAa-PAA-MEA Barhl2 Glut 402
102 DMH-LHA Gsx1 Gaba 254
074 MEA-BST Lhx6 Sp9 Gaba 373
086 MPO-ADP Lhx8 Gaba 241
099 SBPV-PVa Six6 Satb2 Gaba 204
077 CEA-BST Gal Avp Gaba 441
120 MEA Otp Foxp2 Glut 411
107 DMH Hmx2 Gaba 206
129 VMH Nr5a1 Glut 359
085 SI-MPO-LPO Lhx8 Gaba 272
075 MEA-BST Lhx6 Nr2e1 Gaba 340
109 LGv-ZI Otx2 Gaba 290
106 PVpo-VMPO-MPN Hmx2 Gaba 259
128 VMH Fezf1 Glut 363
076 MEA-BST Lhx6 Nfib Gaba 4

In [27]:
folder_name = '_nonmam_cleaned_sssubclass_or_bkgd_06102025/'
if not os.path.isdir('Important_genes/' + folder_name):
    os.mkdir('Important_genes/' + folder_name)
df = pd.DataFrame(data = [qdiffth, diff,difftwo], index = ['qdiffth','diff','diff2']) 
df.to_csv('Important_genes/' + folder_name + '/metadata.csv')
with open('Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(marker_dict, f)

NameError: name 'difftwo' is not defined

In [51]:
folder_name = 'Allen_Institute_PVH'
if not os.path.isdir('Important_genes/' + folder_name):
    os.mkdir('Important_genes/' + folder_name)
df = pd.DataFrame(data = [qdiffth, diff, kn, num_ct_better], index = ['qdiffth', 'diff', 'kn', 'num_ct_better']) 
df.to_csv('Important_genes/' + folder_name + '/metadata.csv')
with open('Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(marker_dict, f)

In [34]:
folder_name = 'MM_cleaned_sssubclass_v3_or_hypobkgd_sigupdate_07042026'
if not os.path.isdir('Important_genes/' + folder_name):
    os.mkdir('Important_genes/' + folder_name)
df = pd.DataFrame(data = [qdiffth,qdiffth2, diff, kn, num_ct_better], index = ['qdiffth','qdiffth2', 'diff', 'kn', 'num_ct_better']) 
df.to_csv('Important_genes/' + folder_name + '/metadata.csv')
with open('Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(marker_dict, f)

In [113]:
folder_name = 'MO_expressedgenes_onepercent_02272026'
if not os.path.isdir('Important_genes/' + folder_name):
    os.mkdir('Important_genes/' + folder_name)
df = pd.DataFrame(data = [th_exp], index = ['th']) 
df.to_csv('Important_genes/' + folder_name + '/metadata.csv')
with open('Important_genes/' + folder_name + '/' + folder_name + '.pkl', 'wb') as f:
    pickle.dump(exp_dict, f)

/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: divide by zero encountered in double_scalars
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: invalid value encountered in double_scalars
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: divide by zero encountered in double_scalars
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: invalid value encountered in double_scalars
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: divide by zero encountered in double_scalars
/scratch/miniconda/lib/python3.7/site-packages/ipykernel_launcher.py:19: RuntimeWarning: invalid value encountered in double_scalars


In [2]:
with open('Important_genes/CJ_cleaned_sssubclass_v3_or_hypobkgd_06012026/CJ_cleaned_sssubclass_v3_or_hypobkgd_06012026.pkl', 'rb') as f:
    test = pickle.load(f)

In [3]:
with open('Important_genes/CJ_cleaned_sssubclass_v3_or_hypobkgd_06082026/CJ_cleaned_sssubclass_v3_or_hypobkgd_06082026.pkl', 'rb') as f:
    marker_dict = pickle.load(f)